In [ ]:
FONT_PATH = './font.bin'
UNILOC_PATH = './uni_loc.json'
LOC2CAM_PATH = './loc2cam.json'

TRN_PATH = "./dl-lab-2-stuff-detection/yolo_dataset/yolo_dataset/train/images"
TST_PATH = "./dl-lab-2-stuff-detection/test_images/test_images"

TRN_META_DST_PATH = './trn_meta'
TST_META_DST_PATH = './tst_meta'

In [ ]:
import re
import json
from datetime import datetime

with open(UNILOC_PATH, 'r', encoding='utf-8') as f:
    UNILOC = json.load(f)
with open(LOC2CAM_PATH, 'r', encoding='utf-8') as f:
    LOC2CAM = json.load(f)

def get_loc_trn_fmt(fname):  # "0209-[LOC]_[TMS].jpg" -> LOC
    return re.split(r'[-_.]', fname)[1]
def get_loc_tst_fmt(fname):  # [LOC]_[TMS].jpg -> LOC
    return fname.split('_')[0]

def parse_datetime(dt: str) -> datetime:
    return datetime.strptime(dt, '%Y%m%d%H%M%S')

In [3]:
ATTN_RECT = (0,0,344,48)
TL_XS = (0,18,36,54, 90,108, 144,162, 198,216, 252,270, 306,324)

In [4]:
from turbojpeg import TurboJPEG, TJPF_GRAY
_jpeg = TurboJPEG()

def turbo_load(img_path, _attn_rect=ATTN_RECT):
    # RES_CPU_RAM = 16656 byte (48x344 uint8)
    with open(img_path, 'rb') as f:
        img_bytes = f.read()
    return _jpeg.decode(
        _jpeg.crop(img_bytes, *_attn_rect),
        pixel_format=TJPF_GRAY
    )


# OR (twise as slow):
# import cv2
# def cv2_load(img_path):
#     # RES_CPU_RAM = 921728 byte (720x1280 uint8)
#     return cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2GRAY)  # type: ignore

In [ ]:
import numpy as np
from typing import Literal

FONT_BOOL = ((np.unpackbits(np.fromfile(FONT_PATH, dtype=np.uint8))[:2800].reshape(10, 20, 14)).astype(bool))

Y_IDX = np.arange(20,40).reshape(1,20,1)
X_IDX_V4 = np.array(TL_XS).reshape(14,1,1) + np.arange(14).reshape(1,1,14)
X_IDX_V2 = np.array(TL_XS).reshape(14,1,1) + 1 + np.arange(14).reshape(1,1,14)

N1 = FONT_BOOL.sum(axis=(1,2), keepdims=True)
N0 = (~FONT_BOOL).sum(axis=(1,2), keepdims=True)
W = np.where(FONT_BOOL, 1.0/N1, -1.0/N0).astype(np.float32)

def _read_v4(img, y_idx=Y_IDX, _x_idx=X_IDX_V4):
    """ except: TRN-7, TRN-16, TRN-26, TST-8 """
    crops = img[y_idx, _x_idx]
    bin_crops = crops > 178
    errors = np.count_nonzero(bin_crops[:, None, :, :] != FONT_BOOL[None, :, :, :], axis=(2, 3))
    best_digits = errors.argmin(axis=1)
    return ''.join(best_digits.astype(str))

def _read_v2(img, _y_idx=Y_IDX, _x_idx=X_IDX_V2):
    crops = img[_y_idx, _x_idx]
    #? crops = np.clip(crops, 95, 235)  # не, не,не,не!!
    scores = np.einsum('nij,mij->nm', crops.astype(np.float32), W)
    best_digits = scores.argmax(axis=1)
    return ''.join(best_digits.astype(str))

def read(img, attn: Literal['v2', 'v4'], alg: Literal['v2', 'v4']):
    if img.ndim == 3: img = img[:,:,0]
    x_idx = X_IDX_V2 if attn == 'v2' else X_IDX_V2
    reader = _read_v4 if attn == 'v4' else _read_v2
    return reader(img, _x_idx=x_idx)

In [ ]:
import os, sys
from tqdm import tqdm

TRN_FNS = sorted(os.listdir(TRN_PATH))
TST_FNS = sorted(os.listdir(TST_PATH))

# _trn_n = len(TRN_FNS)
# _tst_n = len(TST_FNS)
# print(f"#trn={_trn_n}, #tst={_tst_n}, #tot={_tst_n+_trn_n}")
# print("Usage Turbo:")
# print(f"Expect: trn: {_trn_n*16656} byte, tst: {_tst_n*16656} byte")
# print(f"Expect: tot: {(_trn_n+_tst_n)*16656} byte")
# print(f"Expect: tot: {(_trn_n+_tst_n)*16656 // 1024 // 1024} MB")

FNAME2IDX = {}
IDX2FNAME = {}
IMGS = np.zeros((len(TRN_FNS+TST_FNS), 48, 344, 1), dtype=np.uint8)

for i, fn in tqdm(enumerate(TRN_FNS+TST_FNS), total=len(TRN_FNS+TST_FNS)):
    FNAME2IDX[fn] = i
    IDX2FNAME[i] = fn
    root = TRN_PATH if i < len(TRN_FNS) else TST_PATH
    IMGS[i] = turbo_load(f"{root}/{fn}")

100%|██████████| 8362/8362 [00:30<00:00, 273.10it/s]


In [7]:
META = {
    # fname: {"loc": int, "datetime": datetime}
}
_raw_dts = []

for i, fn in tqdm(enumerate(TRN_FNS+TST_FNS), total=len(TRN_FNS)+len(TST_FNS)):
    is_tst = i >= len(TRN_FNS)
    raw_loc = get_loc_tst_fmt(fn) if is_tst else get_loc_trn_fmt(fn)
    loc = UNILOC["TST" if is_tst else "TRN"][raw_loc]
    attn: Literal['v2', 'v4'] = LOC2CAM[loc][-2:].lower()  # type: ignore

    raw_dt = read(IMGS[i], attn=attn, alg='v2')
    _raw_dts.append((i, int(is_tst), fn, loc, raw_dt))
    META[fn] = {
        "loc": loc,
        "datetime": parse_datetime(raw_dt)
    }

100%|██████████| 8362/8362 [00:00<00:00, 14305.13it/s]


In [ ]:
# Проверка ошибок ✨
_errs = set(filter(lambda x: not x[4].startswith('202509'), _raw_dts)) | set(filter(lambda x: x[4][4:6] not in ('02', '09'), _raw_dts))
# _groups = {"trn": {f"{i:02}":[] for i in range(40)}, "tst": {f"{i:02}":[] for i in range(40)}}
# def _get_ts(fn, is_tst):
#     return int(re.split(r'[_.]', fn)[1].replace('-', '')) if is_tst else (int(re.split(r'[-_.]', fn)[2]) // 25)
# for row in _raw_dts:
#     _groups["tst" if is_tst else "trn"][row[3]].append((row[0], _get_ts(row[2], is_tst), parse_datetime(row[-1]), row[-1]))
# for loc, shot in _groups['trn'].items():
#     s = sorted(shot, key=lambda x: x[2])
#     errs = tuple(filter(lambda x: (x[1][1] - x[0][1]) != (x[1][2] - x[0][2]).total_seconds(), zip(s[:-1], s[1:])))
#     _errs |= {(x[0][0], IDX2FNAME[x[0][0]], x[0][-1], "dTS != dPT") for x in errs} | {(x[1][0], IDX2FNAME[x[1][0]], x[1][-1], "dTS != dPT") for x in errs}
# for loc, shot in _groups['tst'].items():
#     s = sorted(shot, key=lambda x: x[2])
#     errs = tuple(filter(lambda x: (x[1][1] > x[0][1]) and ((x[1][2] - x[0][2]).total_seconds() < 0), zip(s[:-1], s[1:])))   
#     _errs |= {(x[0][0], IDX2FNAME[x[0][0]], x[0][-1], "Non-monotonic") for x in errs} | {(x[1][0], IDX2FNAME[x[1][0]], x[1][-1], "Non-monotonic") for x in errs}

_errs

{(785, 0, '0209-16_00923900.jpg', '10', '20250802124931'),
 (818, 0, '0209-16_01041800.jpg', '10', '20250302140836'),
 (828, 0, '0209-16_01055600.jpg', '10', '20250802141748'),
 (883, 0, '0209-16_01237600.jpg', '10', '20250302162243'),
 (5260, 1, '16.1_01-00-37-500.jpg', '39', '11111111111111')}

In [9]:
PATCH_META = {
    '0209-16_00923900.jpg':  '20250902124931',
    '0209-16_01041800.jpg':  '20250902140836',
    '0209-16_01055600.jpg':  '20250902141748',
    '0209-16_01237600.jpg':  '20250902162243',
    '16.1_01-00-37-500.jpg': '20250909124000',
}

for k, v in PATCH_META.items():
    META[k]["datetime"] = parse_datetime(v)

In [ ]:
import pandas as pd

df = pd.DataFrame.from_dict(META, orient='index').reset_index().rename(columns={'index': 'fname'})
df['id'] = df['fname'].map(FNAME2IDX)
df = df[['id', 'fname', 'loc', 'datetime']]

print("COMMON DF"); display(pd.concat([df.head(2), df.tail(2)]))

trn_df = df[df['id']  < len(TRN_FNS)].copy().drop(columns=['id']).reset_index(drop=True)
tst_df = df[df['id'] >= len(TRN_FNS)].copy().drop(columns=['id']).reset_index(drop=True)

print("TRN DF:"); display(pd.concat([trn_df.head(2), trn_df.tail(2)]))
print("TST DF:"); display(pd.concat([tst_df.head(2), tst_df.tail(2)]))

trn_df.to_csv("./trn_meta.csv", index=False)
tst_df.to_csv("./tst_meta.csv", index=False)

COMMON DF


,id,fname,loc,datetime
0,0,0209-10_00864300.jpg,07,2025-09-02 12:16:33
1,1,0209-10_00864400.jpg,07,2025-09-02 12:16:37
8360,8360,8_11-04-47-500.jpg,19,2025-09-02 22:44:07
8361,8361,8_11-05-25-000.jpg,19,2025-09-02 22:49:00


TRN DF:


,fname,loc,datetime
0,0209-10_00864300.jpg,07,2025-09-02 12:16:33
1,0209-10_00864400.jpg,07,2025-09-02 12:16:37
3906,0209-8_00910400.jpg,24,2025-09-02 12:31:00
3907,0209-8_00910500.jpg,24,2025-09-02 12:31:04


TST DF:


,fname,loc,datetime
0,1.1_00-00-00-000.jpg,37,2025-09-09 07:59:58
1,1.1_00-00-12-500.jpg,37,2025-09-09 08:00:43
4452,8_11-04-47-500.jpg,19,2025-09-02 22:44:07
4453,8_11-05-25-000.jpg,19,2025-09-02 22:49:00
